# ResilioChain — Exploratory Data Analysis (Session 2 - Part B)

## Section 2: Grouping and Aggregations

Now that data is loaded and inspected, we move to grouped analysis.
Grouping means: split the data into buckets by a category (product, month,
supplier), apply a calculation inside each bucket (sum, mean, std),
and read the result as a single summary table.

This section answers: who is struggling, when, and why.

### Imports and Data Load

We load both files fresh at the top of every session.
Never assume state carries over from a previous notebook.

In [2]:
import numpy as np
import pandas as pd

inv = pd.read_csv('../../data/inventory_clean.csv')
sup = pd.read_csv('../../data/suppliers.csv')

inv['date'] = pd.to_datetime(inv['date'], format='mixed')

print("inventory_clean shape:", inv.shape)
print("suppliers shape:      ", sup.shape)
print()
print(inv.head())

inventory_clean shape: (1825, 5)
suppliers shape:       (3, 4)

        date product_id  closing_stock  stockout_flag supplier_id
0 2024-01-01       P001            585              0        S001
1 2024-01-02       P001            571              0        S001
2 2024-01-03       P001            557              0        S001
3 2024-01-04       P001            545              0        S001
4 2024-01-05       P001            526              0        S001


### Q1 — Total stockout days per product

PROBLEM:-

We have 1825 rows but we cannot see the big picture from raw rows.
We need to collapse all 365 rows per product into one number:
how many days did each product have zero stock across the full year?
This is our first real business insight from grouping.

In [3]:
stockout_days = inv.groupby('product_id')['stockout_flag'].sum()
print(stockout_days.sort_values(ascending=False))

product_id
P001    125
P004     76
P002     10
P003      0
P005      0
Name: stockout_flag, dtype: int64


**Result**:
  1. P001 : 125 stockout days
  2. P004 :  76 stockout days
  3. P002 :  10 stockout days
  4. P003 :   0 stockout days
  5. P005 :   0 stockout days

-  P001 is the worst offender — out of stock on 125 days out of 365.
- That is 34% of the entire year with nothing on the shelf for that product.
- P001 and P004 share the same supplier (S001, AsiaTech Imports, 14-day lead).
- P003 and P005 never stocked out — both are on faster suppliers.
- The supplier connection is the story here.

### Q2 — Mean closing stock per product

PROBLEM:-

Knowing who runs out is not enough. We also need to know who is sitting
on the most inventory on average. High average stock means high holding cost.
Low average stock means the product is always running lean — one bad day
away from a stockout. Both extremes are a risk.

In [4]:
mean_stock = inv.groupby('product_id')['closing_stock'].mean().round(2)
print(mean_stock.sort_values(ascending=False))

product_id
P005    280.43
P003    250.33
P002    220.71
P004    173.49
P001    138.24
Name: closing_stock, dtype: float64


**Result** (highest to lowest average closing stock):
  1.   P005 : highest average
  2.   P003 : second
  3.   P002 : middle
  4.   P004 : lower
  5.   P001 : lowest average

- P001 has both the highest stockout days (Q1) AND the lowest average
closing stock. This is consistent — a product that runs lean all year
will hit zero more often.
- P003 has zero stockouts AND the highest average stock. It is over-stocked.
High average stock = money sitting idle in the warehouse. That has a cost too.
ResilioChain should flag both extremes: too lean AND too heavy.

### Q3 — Minimum closing stock per product and cross-check with Q1

PROBLEM:-

The mean tells us the average day. The minimum tells us the worst day.
A product can look healthy on average but still have hit zero on its
worst day. We need to know who ever touched the floor — and then
verify that every product that hit zero also has stockout days recorded.
If they do not match, we have a data quality issue.

In [5]:
min_stock = inv.groupby('product_id')['closing_stock'].min()
print("Minimum closing stock per product:")
print(min_stock.sort_values())

Minimum closing stock per product:
product_id
P001     0
P002     0
P004     0
P003    17
P005    30
Name: closing_stock, dtype: int64


In [6]:
print("\nCross-check: products that hit zero vs stockout days recorded:")
zero_products = min_stock[min_stock == 0].index.tolist()
print("Products with min stock = 0:", zero_products)
print()
print("Stockout days for those products:")
print(stockout_days[zero_products])


Cross-check: products that hit zero vs stockout days recorded:
Products with min stock = 0: ['P001', 'P002', 'P004']

Stockout days for those products:
product_id
P001    125
P002     10
P004     76
Name: stockout_flag, dtype: int64


- Products with min closing stock = 0: P001, P002, P004

**Cross-check**:

  P001 : 125 stockout days  — consistent. Hit zero, flag recorded.
  
  P002 :  10 stockout days  — consistent. Hit zero, flag recorded.
  
  P004 :  76 stockout days  — consistent. Hit zero, flag recorded.

- Every product that hit zero stock also has stockout days recorded.
No logical inconsistency. The data is trustworthy on this point.

- P003 and P005 never hit zero — both have non-zero minimums
and zero stockout days. Fully consistent.

### Q4 — Mean, min, max, and std of closing stock in one groupby call

PROBLEM:-

Running four separate groupby calls for mean, min, max, std is
repetitive and slow. Pandas allows you to pass multiple aggregation
functions in a single call using .agg(). This is the professional way
to build summary tables — one clean result instead of four separate outputs.

Standard deviation (std) measures volatility: how wildly stock levels
swing day to day. A high std means the product is being managed
inconsistently — sometimes heavily overstocked, sometimes near empty.

In [7]:
stock_summary = inv.groupby('product_id')['closing_stock'].agg(
    mean_stock='mean',
    min_stock='min',
    max_stock='max',
    std_stock='std'
).round(2)

print(stock_summary.sort_values('std_stock', ascending=False))

            mean_stock  min_stock  max_stock  std_stock
product_id                                             
P004            173.49          0        588     145.71
P001            138.24          0        585     143.12
P002            220.71          0        584     130.89
P005            280.43         30        595     128.42
P003            250.33         17        583     121.96


- P004 has the highest standard deviation (145.71), followed closely by P001 (143.12).
- P004 swings between 0 and 588 units in the same year — that is extreme volatility.
- P003 is the most stable (std = 121.96) and also never stocked out.

**NOTE** The product with the highest std is the most volatile.

High std in closing stock tells you:
  The product swings between very high stock and near-zero stock
  within the same year. This means:
  1. Reorder timing is inconsistent — large batches arrive then
     drain completely before the next order.
  2. Demand may be unpredictable for this product.
  3. The current reorder policy is not working.

A well-managed product has a low std relative to its mean —
it stays in a comfortable band and never swings to extremes.
That is what ResilioChain's reorder recommendations should achieve.

### Q5 — Total stockout days per month across all products

PROBLEM:-

So far we have grouped by product. Now we group by time.
Stockouts may cluster in certain months — seasonal demand spikes,
supplier delays in specific periods, or holiday effects.
Identifying the worst months tells us when the supply chain
is under the most stress across the entire warehouse.

In [8]:
inv['month'] = inv['date'].dt.month

monthly_stockouts = inv.groupby('month')['stockout_flag'].sum()
print(monthly_stockouts.sort_values(ascending=False))

month
12    33
6     24
11    23
7     21
8     20
9     19
3     17
5     15
10    15
2     12
4     12
1      0
Name: stockout_flag, dtype: int64


The output shows total stockout days per month across all products combined.

Look for the month at the top after sort — that is when the warehouse
was under the most stress.

Operational implication:

    Worst month:  December (33 stockout days)
    Best month:   January  (0 stockout days)

- Stockouts build steadily from mid-year and peak in Q4 (Oct-Dec).
- January being zero makes sense — stock was likely built up before year-end.
- The warehouse is most stressed in the second half of the year.

### Q6 — Total stockout days grouped by both month and product_id

PROBLEM:-

Monthly totals from Q5 mix all products together.
A single bad product can dominate the monthly number and hide
the fact that other products were fine that month.
By grouping on two dimensions at once — month AND product —
we can pinpoint exactly which product was causing the damage
in each month. This is called a multi-level groupby.

In [9]:
product_month = inv.groupby(['month', 'product_id'])['stockout_flag'].sum()
print(product_month[product_month > 0].sort_values(ascending=False).head(10))

month  product_id
12     P001          17
6      P001          16
8      P001          13
9      P001          13
7      P004          12
3      P001          12
12     P004          12
11     P001          12
       P004          10
10     P001           9
Name: stockout_flag, dtype: int64


- Worst product-month pair: P001 in December — 17 stockout days in one month.
- P001 appears in 7 of the top 10 worst pairs. It dominates the damage.
- P004 appears in the remaining 3 spots — both S001 products driving all the risk.
- P002, P003, P005 do not appear in the top 10 at all.

### Q7 — Merge with suppliers.csv, then group by supplier_id

PROBLEM:-

inventory_clean.csv only has supplier_id (S001, S002, S003).
To understand which supplier is responsible for the most stockouts,
we first need to JOIN the two tables so each inventory row carries
its supplier's full context.
Then we group by supplier and sum stockout days.
This directly links supplier performance to real warehouse outcomes.

In [13]:
merged = inv.merge(sup, on='supplier_id', how='left')

print("Merged shape:", merged.shape)
print(merged.head())

Merged shape: (1825, 9)
        date product_id  closing_stock  stockout_flag supplier_id  month  \
0 2024-01-01       P001            585              0        S001      1   
1 2024-01-02       P001            571              0        S001      1   
2 2024-01-03       P001            557              0        S001      1   
3 2024-01-04       P001            545              0        S001      1   
4 2024-01-05       P001            526              0        S001      1   

               name  lead_time_days  reliability  
0  AsiaTech Imports              14         0.92  
1  AsiaTech Imports              14         0.92  
2  AsiaTech Imports              14         0.92  
3  AsiaTech Imports              14         0.92  
4  AsiaTech Imports              14         0.92  


In [14]:
supplier_stockouts = merged.groupby('supplier_id')['stockout_flag'].sum()
print(supplier_stockouts.sort_values(ascending=False))

supplier_id
S001    201
S002     10
S003      0
Name: stockout_flag, dtype: int64


**Result**:
  1. S001 (AsiaTech Imports,  14-day lead) : highest stockout days
  2. S002 (EuroGoods Ltd,      7-day lead) : some stockout days
  3. S003 (LocalFast Supply,   3-day lead) : zero stockout days

- S001 is responsible for the vast majority of all stockout days.
- Both products it supplies (P001 and P004) are the worst performers.

**This is the core finding of the entire EDA**:
  Lead time drives stockouts. Long lead time = no time to react
  when stock runs low. The warehouse runs empty while waiting
  two weeks for a delivery that may also arrive late (0.92 reliability).

### Q8 — Total stockout days and mean closing stock per supplier in one call

PROBLEM:-

Q7 told us which supplier has the most stockout days.
But stockout days alone is not the full picture. 
A supplier might have high stockout days but also high average stock —
meaning the product overstocks then crashes.
Or it might have low average stock consistently, meaning it runs lean
and falls into stockout regularly. 
By putting both numbers side by side in one groupby call,
we can see the full pattern together.

In [16]:
supplier_summary = merged.groupby('supplier_id').agg(
    total_stockout_days=('stockout_flag', 'sum'),
    mean_closing_stock=('closing_stock', 'mean')
).round(2)

print(supplier_summary.sort_values('total_stockout_days', ascending=False))

             total_stockout_days  mean_closing_stock
supplier_id                                         
S001                         201              155.87
S002                          10              250.57
S003                           0              250.33


1. S001 (AsiaTech) : 201 stockout days, mean stock = 155.87  — low stock AND most stockouts
2. S002 (EuroGoods):  10 stockout days, mean stock = 250.57  — healthy stock, rare stockouts
3. S003 (LocalFast):   0 stockout days, mean stock = 250.33  — healthy stock, zero stockouts

Pattern:

- S001 has both the most stockout days AND the lowest mean closing stock.
- This is the "runs lean and crashes" pattern — not overstocking then draining.
- The fix for S001 products is earlier reorder triggers, not smaller order quantities.

Now we can compare two dimensions at once per supplier.

- Read the table by asking:
  Does the supplier with the most stockout days ALSO have
  the lowest mean closing stock?

- If yes: the product runs consistently lean and hits zero regularly.
        The fix is to increase reorder quantity or reorder earlier.

- If no (high stockouts but high mean stock): the product overstocks
        massively then drains completely before the next order.
        The fix is more frequent, smaller reorders.

This distinction completely changes the recommended action.
One number alone would give you the wrong prescription.

### Q9 — Stockout rate per product as a percentage

PROBLEM:- 

Raw stockout day counts from Q1 are hard to compare fairly if
products had different amounts of data. In this dataset every
product has exactly 365 days, so raw counts are comparable.
But stockout RATE is the professional metric — it normalises
the count into a percentage that is immediately interpretable:
"P001 is out of stock 34% of the time" is clearer than
"P001 has 125 stockout days."
Rate is also what you would use in any business report or model feature.

In [17]:
total_days = inv.groupby('product_id')['stockout_flag'].count()
total_stockouts = inv.groupby('product_id')['stockout_flag'].sum()

stockout_rate = (total_stockouts / total_days * 100).round(2)
stockout_rate.name = 'stockout_rate_%'

print(stockout_rate.sort_values(ascending=False))
print()
print("Products with stockout rate above 10%:")
print(stockout_rate[stockout_rate > 10])

product_id
P001    34.25
P004    20.82
P002     2.74
P003     0.00
P005     0.00
Name: stockout_rate_%, dtype: float64

Products with stockout rate above 10%:
product_id
P001    34.25
P004    20.82
Name: stockout_rate_%, dtype: float64


**Stockout rates**:
  - P001 : 34.25%  — out of stock more than 1 in 3 days
  - P004 : 20.82%  — out of stock more than 1 in 5 days
  - P002 :  2.74%  — minor issue
  - P003 :  0.00%  — never stocked out
  - P005 :  0.00%  — never stocked out

Products above the 10% threshold: P001 and P004.
Both supplied by S001 (AsiaTech, 14-day lead time).

For a model feature, stockout_rate is more useful than raw count:
  It is scale-independent, percentage-normalised, and directly
  interpretable by any stakeholder without context.

### Q10 — Supplier name, total stockout days, and reliability in one table

PROBLEM:- 

suppliers.csv has a reliability score (probability of on-time delivery).
inventory_clean.csv has actual stockout outcomes.
These are two different measurements of the same thing: supplier performance.
The reliability score is a forecast. The stockout days are the reality.
We need to put them side by side and ask: do they tell the same story,
or does the actual data contradict the stated reliability score?
This is the difference between what a supplier claims and what they deliver.

In [18]:
supplier_perf = merged.groupby('name').agg(
    total_stockout_days=('stockout_flag', 'sum'),
    mean_closing_stock=('closing_stock', 'mean')
).round(2)

supplier_perf = supplier_perf.merge(
    sup[['name', 'reliability']],
    on='name',
    how='left'
)

print(supplier_perf.sort_values('total_stockout_days', ascending=False))

                  name  total_stockout_days  mean_closing_stock  reliability
0     AsiaTech Imports                  201              155.87         0.92
1        EuroGoods Ltd                   10              250.57         0.97
2  LocalFast Supply Co                    0              250.33         0.99


**Final table (sorted by worst stockout days first):**

| Supplier            | Total Stockout Days | Mean Closing Stock | Reliability |
|---------------------|---------------------|--------------------|-------------|
| AsiaTech Imports    | 201                 | 155.87             | 0.92        |
| EuroGoods Ltd       | 10                  | 250.57             | 0.97        |
| LocalFast Supply Co | 0                   | 250.33             | 0.99        |

---

**Do both columns tell the same story?**

YES — and strongly so. The ranking by actual stockout days matches the ranking
by reliability score exactly. Worst reliability = most stockouts.

---

**But the reliability score understates the real problem:**

- 0.92 reliability sounds reasonable — 92% on-time.
- But in reality, AsiaTech's products were stocked out 34% of the time.
- An 8% late delivery rate creates a 34% stockout rate because with a
  14-day lead time, one late delivery leaves the warehouse empty for days
  while waiting for the replacement shipment.

---

**Key analytical insight for ResilioChain:**

Reliability score alone is not enough to evaluate a supplier.
You must combine reliability WITH lead time to understand the true operational risk.
A slightly unreliable fast supplier is far less dangerous than a slightly unreliable slow supplier.